# DBSCAN paso a paso
Recreación didáctica del [GIF de referencia](https://miro.medium.com/v2/1*tc8UF-h0nQqUfLC8-0uInQ.gif) con datos sintéticos. Un anillo forma un solo grupo, y dentro aparecen otros tres grupos que se expanden por separado.

Ejecuta todas las celdas y presiona ▶. Puedes pausar o avanzar un cuadro a la vez. Los círculos sin relleno muestran el radio que se está examinando. La zona sombreada muestra la unión de los radios de los núcleos ya expandidos. Cada punto coloreado pertenece al grupo de ese color.

Dependencias si necesitas instalarlas en Jupyter o Colab: `%pip install numpy matplotlib scikit-learn pillow`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Circle, PathPatch
from matplotlib.path import Path
from IPython.display import HTML, display
from sklearn.cluster import DBSCAN

EPS = 0.12
MIN_SAMPLES = 5  # Incluye el propio punto.
INTERVALO_MS = 180

rng = np.random.default_rng(12)
angulos = np.linspace(0, 2 * np.pi, 150, endpoint=False) + 0.9
anillo = np.column_stack((np.cos(angulos), np.sin(angulos)))
anillo += rng.normal(0, 0.018, size=anillo.shape)

boca = np.column_stack((
    np.linspace(-0.33, 0.33, 30),
    1.1 * np.linspace(-0.33, 0.33, 30)**2 - 0.49,
))
boca += rng.normal(0, 0.025, size=boca.shape)
ojo_izq = rng.normal((-0.40, 0.33), (0.05, 0.05), size=(18, 2))
ojo_der = rng.normal((0.38, 0.32), (0.05, 0.05), size=(18, 2))
densos = np.vstack((anillo, boca, ojo_izq, ojo_der))

# Separamos el ruido para que el ejemplo sea reproducible.
ruido = []
for candidato in rng.uniform(-1.13, 1.13, size=(250, 2)):
    existentes = np.vstack((densos, ruido)) if ruido else densos
    if np.all(np.linalg.norm(existentes - candidato, axis=1) > EPS):
        ruido.append(candidato)
    if len(ruido) == 22:
        break

X = np.vstack((densos, ruido))

## Registrar la expansión
Se calcula el vecindario de cada punto. Un núcleo inicia un grupo o lo expande. Un punto de borde se incorpora cuando es vecino de un núcleo. La cola permite ver cómo avanza la expansión por los dos lados del anillo. La animación guarda un cuadro por cada núcleo expandido y comprueba el resultado con scikit-learn.

In [2]:
def dbscan_animado(X, eps, min_samples):
    distancias = np.linalg.norm(X[:, None] - X[None, :], axis=2)
    vecinos = [np.flatnonzero(fila <= eps) for fila in distancias]
    nucleo = np.array([len(indices) >= min_samples for indices in vecinos])
    etiquetas = np.full(len(X), -1, dtype=int)
    expandidos = np.zeros(len(X), dtype=bool)
    cuadros = [(etiquetas.copy(), expandidos.copy(), None)]
    grupo = 0

    for inicio in range(len(X)):
        if not nucleo[inicio] or etiquetas[inicio] != -1:
            continue

        etiquetas[inicio] = grupo
        pendientes = deque([inicio])

        while pendientes:
            punto = pendientes.popleft()
            nuevos = vecinos[punto][etiquetas[vecinos[punto]] == -1]
            etiquetas[nuevos] = grupo
            pendientes.extend(nuevos[nucleo[nuevos]])
            expandidos[punto] = True
            cuadros.append((etiquetas.copy(), expandidos.copy(), punto))

        grupo += 1

    cuadros.append((etiquetas.copy(), expandidos.copy(), None))
    return etiquetas, cuadros


etiquetas, cuadros = dbscan_animado(X, EPS, MIN_SAMPLES)
referencia = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit_predict(X)
np.testing.assert_array_equal(etiquetas, referencia)
print(f"Grupos: {len(set(etiquetas) - {-1})} | Ruido: {np.sum(etiquetas == -1)} puntos")

Grupos: 4 | Ruido: 22 puntos


## Reproducir la animación
Modifica `EPS`, `MIN_SAMPLES` o `INTERVALO_MS` en la primera celda y vuelve a ejecutar el notebook. El grupo que se expande se distingue por su círculo activo. El texto indica cuántos puntos contiene ese círculo, incluido el núcleo.

In [3]:
COLORES = ["#e5484d", "#4655d7", "#2e9257", "#bf930e"]
fig, ax = plt.subplots(figsize=(6.6, 6), dpi=85)
fig.subplots_adjust(top=0.81, bottom=0.04)
fig.suptitle("DBSCAN · Expansión de grupos", fontsize=17, fontweight="bold", y=0.97)

margen = EPS + 0.05
x_min, y_min = X.min(axis=0) - margen
x_max, y_max = X.max(axis=0) + margen
# Preparamos los círculos para reutilizarlos en todos los cuadros.
circulos = []
for centro in X:
    circulo = Circle(centro, EPS)
    circulos.append(circulo.get_path().transformed(circulo.get_transform()))


def dibujar(indice):
    labels, expandidos, activo = cuadros[indice]
    ax.clear()
    ax.set(xlim=(x_min, x_max), ylim=(y_min, y_max), aspect="equal")
    ax.axis("off")

    sin_grupo = labels == -1
    ax.scatter(*X[sin_grupo].T, s=11, facecolors="none", edgecolors="#747b83",
               linewidths=0.65, zorder=3)
    for grupo in np.unique(labels[~sin_grupo]):
        color = COLORES[grupo % len(COLORES)]
        miembros = labels == grupo
        indices = np.flatnonzero(miembros & expandidos)
        if len(indices):
            union = Path.make_compound_path(*(circulos[i] for i in indices))
            ax.add_patch(PathPatch(union, facecolor=color, edgecolor="none", alpha=0.30))
        ax.scatter(*X[miembros].T, s=14, color=color, edgecolors="#252c34",
                   linewidths=0.35, zorder=4)

    if activo is not None:
        grupo = labels[activo]
        color = COLORES[grupo % len(COLORES)]
        ax.add_patch(PathPatch(circulos[activo], fill=False, edgecolor=color,
                               linewidth=2, zorder=5))
        cantidad = np.count_nonzero(np.linalg.norm(X - X[activo], axis=1) <= EPS)
        estado = f"Grupo {grupo + 1} · Núcleo con {cantidad} puntos en su radio"
    elif indice == 0:
        estado = "Puntos antes de la expansión"
    else:
        estado = f"Resultado · {len(set(labels) - {-1})} grupos · {sin_grupo.sum()} puntos de ruido"
    ax.set_title(f"ε = {EPS:g}     ·     min_samples = {MIN_SAMPLES}\n{estado}",
                 fontsize=11, linespacing=1.5, pad=10)


# Repetimos el cuadro final para que pueda leerse antes de volver al inicio.
secuencia = [0] * 5 + list(range(1, len(cuadros) - 1)) + [len(cuadros) - 1] * 13
animacion = FuncAnimation(fig, dibujar, frames=secuencia,
                          interval=INTERVALO_MS, repeat=True)
plt.close(fig)
display(HTML(animacion.to_jshtml(default_mode="once")))

In [4]:
# Opcional: quita el comentario para guardar un GIF en el directorio del notebook.
animacion.save("dbscan_paso_a_paso.gif", writer=PillowWriter(fps=1000 / INTERVALO_MS))